# 01 · Data Quality Report

**Goal:** check that the Gold-layer data is clean and consistent before any analysis is built on it.

**Source:** `gold.report_customers`, `gold.report_products`, `gold.fact_sales` (SQL Server, `DataWarehouse` database)

**Checks:** missing values · duplicates · referential integrity · business rules · average order value (AOV) consistency · outliers · age sanity · quick visuals

**Output:** audit tables and cleaned data in `output/`

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import config

sns.set_style("whitegrid")
os.makedirs("output", exist_ok=True)

## 2. Load data from the Gold layer

Connection details are kept in `config.py`.

In [2]:
engine = create_engine(config.get_conn_url())

df_customers = pd.read_sql("select * from gold.report_customers", engine)
df_products = pd.read_sql("select * from gold.report_products", engine)
df_sales = pd.read_sql("select * from gold.fact_sales", engine)

## 3. Structure and data types

In [3]:
print(df_customers.info(), df_products.info(), df_sales.info())

<class 'pandas.DataFrame'>
RangeIndex: 18482 entries, 0 to 18481
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_key       18482 non-null  int64  
 1   customer_number    18482 non-null  str    
 2   customer_name      18482 non-null  str    
 3   age                18465 non-null  float64
 4   age_group          18482 non-null  str    
 5   customer_segments  18482 non-null  str    
 6   recency            18482 non-null  int64  
 7   total_order        18482 non-null  int64  
 8   total_sales        18482 non-null  int64  
 9   avg_order_value    18482 non-null  int64  
 10  avg_monthly_sales  18482 non-null  int64  
 11  total_quantity     18482 non-null  int64  
 12  total_products     18482 non-null  int64  
 13  last_orderdate     18482 non-null  object 
 14  lifespan           18482 non-null  int64  
dtypes: float64(1), int64(9), object(1), str(4)
memory usage: 2.1+ MB
<class 'pandas.D

Customers: 18,482 rows (only `age` has missing values: 17). Products: 130 rows. Sales: 60,398 rows, of which 19 have no `order_date`.

## 4. Fill missing order dates

Sales rows without an `order_date` are matched to customers. When the customer placed exactly one order, the missing date is filled with that customer's `last_orderdate` from `report_customers`.

In [4]:
isna_cs_key = df_sales.loc[df_sales["order_date"].isna(), "customer_key"]

In [5]:
# Columns shared between tables (used to bring the order date across)
com_col1 = set(df_sales) & set(df_products)
com_col2 = set(df_sales) & set(df_customers)
com_col3 = set(df_customers) & set(df_products)
print(com_col1)
print(com_col2)
print(com_col3)

{'product_key'}
{'customer_key'}
{'total_quantity', 'total_sales', 'lifespan'}


In [6]:
df_customers[
    (df_customers["customer_key"].isin(isna_cs_key))
    & (df_customers["total_order"] == 1)
].count()

customer_key         9
customer_number      9
customer_name        9
age                  9
age_group            9
customer_segments    9
recency              9
total_order          9
total_sales          9
avg_order_value      9
avg_monthly_sales    9
total_quantity       9
total_products       9
last_orderdate       9
lifespan             9
dtype: int64

In [7]:
# Step 1 (check): how many missing dates can be fixed this way?
# df_customers[(df_customers['customer_key'].isin(isna_cs_key))&(df_customers['total_order']==1)].count()
# Step 2 (fix): map each single-order customer to their last order date
single_order_cust = df_customers[df_customers["total_order"] == 1]
date_mapping = single_order_cust.set_index("customer_key")["last_orderdate"]
# Step 3: fill the missing order dates in the sales table using this map
missing_mask = df_sales["order_date"].isnull()
df_sales.loc[missing_mask, "order_date"] = df_sales.loc[
    missing_mask, "customer_key"
].map(date_mapping)

In [8]:
df_sales["order_date"].isna().sum()

np.int64(8)

The fix filled 11 of the 19 missing order dates, so 8 sales rows still have no `order_date`. Notebooks 02-08 read `gold.fact_sales` with `order_date IS NOT NULL`, so these rows are excluded from them.

## 5. Convert date columns

Every column with `date` in its name is converted to a datetime type.

In [9]:
for df in [df_customers, df_products, df_sales]:
    for col in [c for c in df.columns if "date" in c.lower()]:
        df[col] = pd.to_datetime(df[col], errors="coerce")

The latest order date is used as the reference date of the analysis, instead of the system date.

In [10]:
# print(df_sales['order_date'].max())#
# print(df_customers['last_orderdate'].max())
analysis_date = df_sales["order_date"].max()

## 6. Products schema and duplicate checks

In [11]:
# df_sales.info()
# df_customers.info()
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype        
---  ------               --------------  -----        
 0   product_name         130 non-null    str          
 1   product_key          130 non-null    int64        
 2   category             130 non-null    str          
 3   subcategory          130 non-null    str          
 4   cost                 130 non-null    int64        
 5   total_orders         130 non-null    int64        
 6   total_sales          130 non-null    int64        
 7   lifespan             130 non-null    int64        
 8   recency_in_months    130 non-null    int64        
 9   product_segemnt      130 non-null    str          
 10  last_sales_date      130 non-null    datetime64[s]
 11  total_customers      130 non-null    int64        
 12  total_quantity       130 non-null    int64        
 13  avg_selling_price    130 non-null    float64      
 14  avg_o

In [12]:
print(df_products.duplicated().sum())
print(df_customers.duplicated().sum())
print(df_sales.duplicated().sum())
print("Duplicate full customer rows:", df_customers.duplicated().sum())
print("Duplicate customer_key:", df_customers["customer_key"].duplicated().sum())
print("Duplicate customer_number:", df_customers["customer_number"].duplicated().sum())

print("Duplicate full product rows:", df_products.duplicated().sum())
print("Duplicate product_key:", df_products["product_key"].duplicated().sum())

0
0
0
Duplicate full customer rows: 0
Duplicate customer_key: 0
Duplicate customer_number: 0
Duplicate full product rows: 0
Duplicate product_key: 0


No duplicate rows and no duplicate keys in the customer and product tables.

## 7. Referential integrity

Do all customer and product keys used in the sales table exist in the report tables?

In [13]:
customer_keys = set(df_customers["customer_key"].dropna().unique())
product_keys = set(df_products["product_key"].dropna().unique())

missing_customer_keys = sorted(
    set(df_sales["customer_key"].dropna().unique()) - customer_keys
)
missing_product_keys = sorted(
    set(df_sales["product_key"].dropna().unique()) - product_keys
)

print(
    "Sales rows with customer keys missing from report_customers:",
    df_sales["customer_key"].isin(missing_customer_keys).sum(),
)
print("Distinct missing customer keys:", len(missing_customer_keys))
print("Distinct missing product keys:", len(missing_product_keys))

Sales rows with customer keys missing from report_customers: 3
Distinct missing customer keys: 2
Distinct missing product keys: 0


3 sales rows point to 2 customer keys that are not in `report_customers` (probably customers whose only orders have no order date, which that view filters out). All product keys match.

## 8. Business-rule checks

In [14]:
sales_rules = {
    "sales_amount < 0": int((df_sales["sales_amount"] < 0).sum()),
    "quantity <= 0": int((df_sales["quantity"] <= 0).sum()),
    "price < 0": int((df_sales["price"] < 0).sum()),
    "order_date null": int(df_sales["order_date"].isna().sum()),
}
print(pd.Series(sales_rules).sort_values(ascending=False))

order_date null     8
sales_amount < 0    0
quantity <= 0       0
price < 0           0
dtype: int64


No negative sales or prices and no zero or negative quantities. The only issue is the 8 rows with a missing order date.

## 9. Average order value (AOV) consistency

The AOV reported by SQL is compared with a recalculated value (`total_sales / total_order`).

16,677 customers have a reported AOV equal to the recalculated one; 1,805 differ by 0.01 or more. The row-level comparison is exported in `aov_consistency_check.csv`.

## 10. Outliers in customer sales (IQR method)

In [15]:
Q1 = df_customers["total_sales"].quantile(0.25)
Q3 = df_customers["total_sales"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
uper = Q3 + 1.5 * IQR

sales_outliers = df_customers[
    (df_customers["total_sales"] < lower) | (df_customers["total_sales"] > uper)
]
print(f"iqr lower bound:{lower:.2f}")
print(f"iqr upper bound:{uper:.2f}")
print(f"outliers cusotmers: {len(sales_outliers)}/{len(df_customers)}")

iqr lower bound:-3641.50
iqr upper bound:6202.50
outliers cusotmers: 676/18482


IQR bounds are −3,641.50 and 6,202.50, so 676 of 18,482 customers (3.7%) are high-spend outliers. The lower bound is negative, so only the upper side can flag a customer.

## 11. Age sanity check

In [16]:
age_summary = {
    "min_age": float(df_customers["age"].min()),
    "max_age": np.nanmax(df_customers["age"]),
    "mean age": df_customers["age"].mean(),
    "null or empty vales": df_customers["age"].isnull().sum(),
    "0 age check": (df_customers["age"] <= 0).sum(),
    "above 100 how many": (df_customers["age"] >= 100).sum(),
}

print(pd.Series(age_summary))

min_age                 40.000000
max_age                110.000000
mean age                57.177471
null or empty vales     17.000000
0 age check              0.000000
above 100 how many      22.000000
dtype: float64


Ages run from 40 to 110 (mean 57.2). 17 are missing and 22 are 100 or above. Age is calculated in SQL against today's date, so it is higher than the customer's age at the time of the 2010-2014 orders. Use age groups as a relative grouping.

## 12. Quick visual check

In [17]:
plt.figure(figsize=(8, 5))
sns.histplot(df_customers["total_sales"].dropna(), bins=50)
plt.title("Customer Total Sales Distribution")
plt.xlabel("Total Sales")
plt.ylabel("Customers")
plt.tight_layout()
plt.savefig("output/01_total_sales_distribution.png", dpi=120)
plt.close()

seg_rev = df_customers.groupby("customer_segments")["total_sales"].sum().sort_values()
plt.figure(figsize=(7, 5))
seg_rev.plot(kind="barh")
plt.title("Total Revenue by Customer Segment")
plt.xlabel("Total Sales in 100 million")
plt.tight_layout()
plt.savefig("output/02_revenue_by_segment.png", dpi=120)
plt.close()

![Customer total sales distribution](output/01_total_sales_distribution.png)

![Revenue by customer segment](output/02_revenue_by_segment.png)

## 13. Missing-value report

In [18]:
null_report = pd.DataFrame(
    {
        "missing_count": df_customers.isna().sum(),
        "missing_pct": (100 * df_customers.isna().mean()).round(2),
    }
).sort_values("missing_count", ascending=False)

print(null_report)

                   missing_count  missing_pct
age                           17         0.09
customer_number                0         0.00
customer_key                   0         0.00
customer_name                  0         0.00
age_group                      0         0.00
customer_segments              0         0.00
recency                        0         0.00
total_order                    0         0.00
total_sales                    0         0.00
avg_order_value                0         0.00
avg_monthly_sales              0         0.00
total_quantity                 0         0.00
total_products                 0         0.00
last_orderdate                 0         0.00
lifespan                       0         0.00


Only `age` has missing values (17 customers, 0.09%).

## 14. Save audit outputs

In [20]:
null_report.to_csv("output/data_quality_null_report.csv")
sales_outliers.to_csv("output/customer_sales_outliers.csv", index=False)
df_customers.to_csv("output/df_customers_clean_v2.csv", index=False)
df_sales.to_csv("output/df_sales_clean_v2.csv", index=False)
df_products.to_csv("output/df_products_clean", index=False)

print("Data quality audit complete. Check the output/ folder.")

Data quality audit complete. Check the output/ folder.
